# Qwen2.5-7B VHE QLoRA

Train a new LoRA adapter from the pretrained Qwen2.5-7B model. Evaluate the full DEV split after every epoch.

## 1. Runtime and experiment settings

In [ ]:
import time
from pathlib import Path
import json, shutil

deadline = time.monotonic() + 5 * 3600
%pip install -q "transformers==4.57.6" "peft==0.22.0" "bitsandbytes>=0.46.1" accelerate datasets
import torch

assert torch.cuda.is_available()
input_root = Path("/kaggle/input")
data_root = next(p.parent for p in input_root.rglob("train.jsonl") if "vhe-adapted" in p.parts)
metrics_root = next(p for p in input_root.rglob("metrics") if (p / "metrics" / "evaluator.py").is_file())
model_path = next(p.parent for p in input_root.rglob("config.json") if p.parent.name == "1" and "qwen2.5" in p.parts)
MAX_EPOCHS = 10
STEPS_PER_EPOCH = 409
DEV_PREDICT_BATCH_SIZE = 4
DEV_MAX_NEW_TOKENS = 256
SEED = 42
required = ("adapter_config.json", "adapter_model.safetensors", "optimizer.pt",
            "scheduler.pt", "rng_state.pth", "scaler.pt", "trainer_state.json")

class DevBudgetReached(Exception):
    pass


## 2. Imports

Import model, dataset, QLoRA, and evaluation utilities.

In [ ]:
from pathlib import Path
import json
import gc
import sys
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
    Trainer,
    TrainerCallback,
)
from transformers.trainer_utils import EvalLoopOutput
from transformers.trainer_callback import TrainerState

from peft import (
    prepare_model_for_kbit_training,
    get_peft_model,
    LoraConfig,
)
set_seed(SEED)


## 3. Load tokenizer and base model

Load the Qwen2.5-7B base model attached to the Kaggle notebook in 4-bit mode for T4 GPUs.

In [ ]:
print("Model path:", model_path)

# 4-bit NF4 quantization for QLoRA.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

# Split the quantized base model across the available GPUs.
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepare the quantized model for LoRA training.
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

# Required for gradient checkpointing during training.
model.config.use_cache = False

print("Model loaded!")
print("Device:", model.device)

## 4. Tokenize one example

Concatenate input and target tokens while computing loss only on the target and EOS token.

In [ ]:
def tokenize_example(example):
    input_text = example["input_text"]
    target_text = example["target_text"]

    input_tokens = tokenizer(
        input_text,
        add_special_tokens=True,
    )

    target_tokens = tokenizer(
        target_text,
        add_special_tokens=False,
    )

    input_ids = input_tokens["input_ids"]
    target_ids = target_tokens["input_ids"]

    input_ids = (
        input_tokens["input_ids"]
        + target_ids
        + [tokenizer.eos_token_id]
    )

    labels = (
        [-100] * len(input_tokens["input_ids"])
        + target_ids
        + [tokenizer.eos_token_id]
    )

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids),
    }

## 5. Format event-extraction data

Convert every sample into an `input_text` / JSON `target_text` pair.

In [ ]:
def format_example(example):
    text = example["text"]

    target = {
        "events": example["events"]
    }

    return {
        "input_text": text,
        "target_text": json.dumps(
            target,
            ensure_ascii=False,
            separators=(",", ":")
        )
    }

## 6. Data collator

Pad each batch dynamically and mask label padding with `-100`.

In [ ]:
def ee_data_collator(features):
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    for f in features:
        pad_length = max_length - len(f["input_ids"])

        input_ids.append(
            f["input_ids"] + [tokenizer.pad_token_id] * pad_length
        )

        attention_mask.append(
            f["attention_mask"] + [0] * pad_length
        )

        labels.append(
            f["labels"] + [-100] * pad_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

## 7. Configure and apply LoRA

Attach LoRA adapters to the attention and MLP projections.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)

## 8. Load and preprocess the dataset

Load the VHE TRAIN and DEV splits.

In [ ]:
# dataset = load_dataset(
#     "json",
#     data_files={
#         "train": "## Data path/train.jsonl",
#         "dev": "## Data path/dev.jsonl",
#     },
# )

dataset = load_dataset(
    "json",
    data_files={
        "train": str(data_root / "train.jsonl"),
        "dev": str(data_root / "dev.jsonl"),
    },
)

# Raw datasets
train_raw = dataset["train"]
dev_raw = dataset["dev"]

# Formatted datasets
train_formatted = train_raw.map(format_example)
dev_formatted = dev_raw.map(format_example)

# Tokenized datasets for training
train_dataset = train_formatted.map(
    tokenize_example,
    remove_columns=train_formatted.column_names,
)

# Tokenized dataset for validation loss.
dev_dataset = dev_formatted.map(
    tokenize_example,
    remove_columns=dev_formatted.column_names,
)

# Raw/formatted dataset for generation evaluation
dev_generation_dataset = dev_formatted
assert len(dev_dataset) == 406


## 9. DEV metrics

After every epoch, generate unconstrained greedy predictions on the full DEV set and compute TI, TC, AI, AC, and average F1.

In [ ]:
# sys.path.insert(0, "## Metrics path")

sys.path.insert(0, str(metrics_root))

from metrics.evaluator import Evaluator

metric_calculator = Evaluator()

## Dual-decoding trainer and checkpoint selector

Use deterministic greedy decoding without constrained decoding. Event types, roles, text, and spans remain model decisions.

In [ ]:
EVENT_OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "events": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "event_type": {"type": "string"},
                    "trigger": {
                        "type": "array",
                        "minItems": 1,
                        "items": {
                            "type": "object",
                            "properties": {
                                "text": {"type": "string"},
                                "span": {
                                    "type": "array",
                                    "items": {"type": "integer"},
                                    "minItems": 2,
                                    "maxItems": 2,
                                },
                            },
                            "required": ["text", "span"],
                            "additionalProperties": False,
                        },
                    },
                    "arguments": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "role": {"type": "string"},
                                "mentions": {
                                    "type": "array",
                                    "items": {
                                        "type": "object",
                                        "properties": {
                                            "text": {"type": "string"},
                                            "span": {
                                                "type": "array",
                                                "items": {"type": "integer"},
                                                "minItems": 2,
                                                "maxItems": 2,
                                            },
                                        },
                                        "required": ["text", "span"],
                                        "additionalProperties": False,
                                    },
                                },
                            },
                            "required": ["role", "mentions"],
                            "additionalProperties": False,
                        },
                    },
                },
                "required": ["event_type", "trigger", "arguments"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["events"],
    "additionalProperties": False,
}

RUN_ROOT = Path("/kaggle/working/vhe-fresh")
PREDICTION_ROOT = RUN_ROOT / "dev_predictions"
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)


class BaselineGenerationTrainer(Trainer):
    def __init__(
        self,
        generation_dataset,
        tokenizer,
        evaluator,
        prediction_root,
        max_new_tokens=256,
        **kwargs,
    ):
        super().__init__(processing_class=tokenizer, **kwargs)
        self.generation_dataset = generation_dataset
        self.evaluator = evaluator
        self.prediction_root = Path(prediction_root)
        self.max_new_tokens = max_new_tokens

    def evaluate(self, *args, **kwargs):
        if self.is_in_train:
            self._save_checkpoint(self.model, trial=None)
            publish_checkpoint(self.state, dev_pending=True)
        return super().evaluate(*args, **kwargs)

    def _generate_baseline(self):
        partial = self.prediction_root / f"epoch-{round(self.state.epoch)}-baseline.partial.jsonl"
        records = [json.loads(line) for line in partial.read_text().splitlines()] if partial.exists() else []
        assert len(records) <= len(self.generation_dataset)
        assert all(row["index"] == index for index, row in enumerate(records))
        predictions = [row["prediction"] for row in records]
        golds = [row["gold"] for row in records]
        print(f"DEV_RESUME {len(records)}/{len(self.generation_dataset)}", flush=True)
        tokenizer = self.processing_class
        previous_padding_side = tokenizer.padding_side
        tokenizer.padding_side = "left"
        try:
            with partial.open("a", encoding="utf-8", buffering=1) as stream, torch.inference_mode():
                for index in range(len(records), len(self.generation_dataset), DEV_PREDICT_BATCH_SIZE):
                    if time.monotonic() >= deadline:
                        raise DevBudgetReached()
                    if index % 32 == 0:
                        print(f"DEV generation {index}/{len(self.generation_dataset)}", flush=True)
                    batch = [self.generation_dataset[i] for i in range(
                        index, min(index + DEV_PREDICT_BATCH_SIZE, len(self.generation_dataset)))]
                    inputs = tokenizer([row["input_text"] for row in batch], padding=True,
                                       return_tensors="pt", add_special_tokens=True).to(self.model.device)
                    generated = self.model.generate(
                        **inputs, max_new_tokens=self.max_new_tokens, do_sample=False, num_beams=1,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                    )
                    decoded = tokenizer.batch_decode(
                        generated[:, inputs.input_ids.shape[1]:], skip_special_tokens=True,
                        clean_up_tokenization_spaces=False,
                    )
                    for offset, (example, prediction) in enumerate(zip(batch, decoded)):
                        gold = {"events": example["events"]}
                        record = {"index": index + offset, "text": example["text"],
                                  "gold": gold, "prediction": prediction}
                        predictions.append(prediction)
                        golds.append(gold)
                        records.append(record)
                        stream.write(json.dumps(record, ensure_ascii=False) + "\n")

        finally:
            tokenizer.padding_side = previous_padding_side

        epoch = int(round(self.state.epoch or 0))
        output_path = self.prediction_root / f"epoch-{epoch}-baseline.jsonl"
        with output_path.open("w", encoding="utf-8") as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")

        results = self.evaluator.evaluate_on_all_sample(predictions, golds)
        valid_count = 0
        for prediction in predictions:
            try:
                parsed = json.loads(prediction)
                valid_count += int(
                    isinstance(parsed, dict)
                    and isinstance(parsed.get("events"), list)
                )
            except (json.JSONDecodeError, TypeError):
                pass
        results["valid_json_rate"] = valid_count / max(1, len(predictions))
        return results

    def evaluation_loop(
        self,
        dataloader,
        description,
        prediction_loss_only=None,
        ignore_keys=None,
        metric_key_prefix="eval",
    ):
        output = super().evaluation_loop(
            dataloader,
            description,
            True,
            ignore_keys,
            metric_key_prefix,
        )
        was_training = self.model.training
        old_use_cache = self.model.config.use_cache
        self.model.eval()
        self.model.config.use_cache = True
        try:
            baseline_results = self._generate_baseline()
        finally:
            self.model.config.use_cache = old_use_cache
            if was_training:
                self.model.train()

        metrics = dict(output.metrics)
        for mode, results in {"baseline": baseline_results}.items():
            metrics[f"{metric_key_prefix}_{mode}_avg_f1"] = results["avg_f1"]
            metrics[f"{metric_key_prefix}_{mode}_valid_json_rate"] = (
                results["valid_json_rate"]
            )
            for name in ("TI", "TC", "AI", "AC"):
                metrics[f"{metric_key_prefix}_{mode}_{name.lower()}_f1"] = (
                    results[name]["f1"]
                )
        self.baseline_generation_results = baseline_results
        return EvalLoopOutput(
            predictions=output.predictions,
            label_ids=output.label_ids,
            metrics=metrics,
            num_samples=output.num_samples,
        )


class BaselineEarlyStoppingCallback(TrainerCallback):
    def __init__(
        self,
        output_root,
        tokenizer,
        patience=3,
        threshold=0.002,
        grace_epochs=2,
        initial_state=None,
    ):
        self.output_root = Path(output_root)
        self.tokenizer = tokenizer
        self.patience = patience
        self.threshold = threshold
        self.grace_epochs = grace_epochs
        self.best = {"baseline": float("-inf")}
        self.early_stop_reference = {
            "baseline": float("-inf"),
        }
        self.best_epoch = {"baseline": None}
        self.stale_epochs = {"baseline": 0}
        if initial_state is not None:
            self.best["baseline"] = initial_state["best_avg_f1"]
            self.early_stop_reference["baseline"] = initial_state["best_avg_f1"]
            self.best_epoch["baseline"] = initial_state["best_epoch"]
            self.stale_epochs["baseline"] = initial_state["stale_epochs"]

    def _save_state(self):
        payload = {
            "best_avg_f1": self.best,
            "best_epoch": self.best_epoch,
            "early_stop_reference": self.early_stop_reference,
            "stale_epochs": self.stale_epochs,
            "patience": self.patience,
            "threshold": self.threshold,
            "grace_epochs": self.grace_epochs,
        }
        self.output_root.mkdir(parents=True, exist_ok=True)
        (self.output_root / "best_state.json").write_text(
            json.dumps(payload, indent=2),
            encoding="utf-8",
        )

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if not state.is_world_process_zero:
            return control

        epoch = int(round(state.epoch or 0))
        for mode in ("baseline",):
            key = f"eval_{mode}_avg_f1"
            if metrics is None or key not in metrics:
                raise KeyError(f"Missing DEV selection metric: {key}")
            score = float(metrics[key])
            if score > self.best[mode]:
                self.best[mode] = score
                self.best_epoch[mode] = epoch
                best_dir = self.output_root / f"best-{mode}"
                model.save_pretrained(best_dir)
                self.tokenizer.save_pretrained(best_dir)

            significant_improvement = (
                score > self.early_stop_reference[mode] + self.threshold
            )
            if significant_improvement:
                self.early_stop_reference[mode] = score
                self.stale_epochs[mode] = 0
            elif epoch > self.grace_epochs:
                self.stale_epochs[mode] += 1
            else:
                self.stale_epochs[mode] = 0

        self._save_state()
        if (
            epoch > self.grace_epochs
            and self.stale_epochs["baseline"] >= self.patience
        ):
            control.should_training_stop = True
        return control

## 10. TrainingArguments

Evaluate and save after every epoch. The callback below selects the best baseline model using DEV average F1.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(RUN_ROOT / "checkpoints"),

    num_train_epochs=MAX_EPOCHS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    logging_strategy="steps",
    logging_steps=25,
    logging_first_step=True,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    warmup_steps=0,
    seed=SEED,

    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=4,

    load_best_model_at_end=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,
    report_to="none",
)


Epochs 1–2 are grace epochs: they are evaluated and can become best checkpoints, but they do not consume patience. Starting at epoch 3, training stops when baseline DEV average F1 does not improve by at least 0.002 for three consecutive evaluations.

In [ ]:
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.002
EARLY_STOPPING_GRACE_EPOCHS = 2

## 11. Clear caches and inspect GPU memory

Clear Python/CUDA caches and print memory usage for every visible GPU.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
        f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved"
    )

## 12. Initialize the trainer and verify DEV batching

Use the same full DEV set for validation loss and baseline generation.

In [ ]:
pointer = RUN_ROOT / "resume-state.json"
source_pointers = list(input_root.rglob("vhe-fresh/resume-state.json"))
assert len(source_pointers) <= 1, "Attach only one continuation output."
resume = None
checkpoint = None
if source_pointers:
    source = source_pointers[0].parent
    resume = json.loads(source_pointers[0].read_text())
    checkpoint = RUN_ROOT / "checkpoints" / f"checkpoint-{resume['step']}"
    shutil.copytree(source / "checkpoints" / checkpoint.name, checkpoint)
    assert all((checkpoint / name).stat().st_size for name in required)
    assert json.loads((checkpoint / "trainer_state.json").read_text())["global_step"] == resume["step"]
    shutil.copy2(source_pointers[0], pointer)
    if (source / "best_state.json").is_file():
        shutil.copy2(source / "best_state.json", RUN_ROOT / "best_state.json")
    if (source / "best-baseline").is_dir():
        shutil.copytree(source / "best-baseline", RUN_ROOT / "best-baseline")
    if resume.get("dev_pending"):
        partial_name = f"epoch-{round(resume['epoch'])}-baseline.partial.jsonl"
        partial = source / "dev_predictions" / partial_name
        if partial.is_file():
            shutil.copy2(partial, PREDICTION_ROOT / partial_name)
selection = resume.get("selection") if resume else None

def publish_checkpoint(state, dev_pending=False):
    saved = RUN_ROOT / "checkpoints" / f"checkpoint-{state.global_step}"
    assert all((saved / name).stat().st_size for name in required)
    selection_file = RUN_ROOT / "best_state.json"
    selected = json.loads(selection_file.read_text()) if selection_file.exists() else None
    epoch = round(state.epoch)
    done = not dev_pending and (RUN_ROOT / f"epoch-{epoch}-dev-results.json").is_file()
    if done:
        (PREDICTION_ROOT / f"epoch-{epoch}-baseline.partial.jsonl").unlink(missing_ok=True)
    pending = pointer.with_suffix(".tmp")
    pending.write_text(json.dumps({"step": state.global_step, "epoch": state.epoch,
                                   "selection": selected, "dev_pending": dev_pending}, indent=2))
    pending.replace(pointer)
    print("CHECKPOINT_COMMITTED", saved.name, state.epoch, "dev_pending", dev_pending, flush=True)

baseline_selection_callback = BaselineEarlyStoppingCallback(
    output_root=RUN_ROOT, tokenizer=tokenizer,
    patience=EARLY_STOPPING_PATIENCE, threshold=EARLY_STOPPING_THRESHOLD,
    grace_epochs=EARLY_STOPPING_GRACE_EPOCHS,
    initial_state={k: selection[k]["baseline"] for k in ("best_avg_f1", "best_epoch", "stale_epochs")}
                  if selection else None,
)
if selection:
    baseline_selection_callback.early_stop_reference = selection["early_stop_reference"].copy()

class Progress(TrainerCallback):
    def on_train_begin(self, args, state, control, optimizer, lr_scheduler, **kwargs):
        for group in optimizer.param_groups:
            group["lr"] = args.learning_rate
            group["initial_lr"] = args.learning_rate
        lr_scheduler.base_lrs = [args.learning_rate] * len(optimizer.param_groups)
        lr_scheduler._last_lr = lr_scheduler.base_lrs.copy()
        state.save_steps = args.save_steps
        state.logging_steps = args.logging_steps
        print(f"TRAIN_START step={state.global_step} constant_lr={args.learning_rate}", flush=True)

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.started = time.monotonic()

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % STEPS_PER_EPOCH == STEPS_PER_EPOCH - 1:
            control.should_save = True
        if time.monotonic() >= deadline:
            control.should_training_stop = True
            control.should_save = True
            control.should_evaluate = False
        return control

    def on_epoch_end(self, args, state, control, **kwargs):
        if time.monotonic() >= deadline:
            control.should_evaluate = False
        else:
            elapsed = time.monotonic() - self.started
            control.should_save = True
            print(f"TRAIN_COMPLETE epoch={state.epoch:.4f} train_seconds={elapsed:.2f}", flush=True)
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        epoch = round(state.epoch)
        (RUN_ROOT / f"epoch-{epoch}-dev-results.json").write_text(json.dumps(metrics, indent=2))
        print("DEV_COMPLETE", json.dumps(metrics), flush=True)
        return control

    def on_save(self, args, state, control, **kwargs):
        publish_checkpoint(state)


trainer = BaselineGenerationTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    generation_dataset=dev_generation_dataset,
    tokenizer=tokenizer,
    evaluator=metric_calculator,
    prediction_root=PREDICTION_ROOT,
    max_new_tokens=DEV_MAX_NEW_TOKENS,
    data_collator=ee_data_collator,
    callbacks=[baseline_selection_callback, Progress()],
)

batch = next(iter(trainer.get_eval_dataloader()))
print("DEV samples:", len(dev_dataset))
print("DEV batch shape:", tuple(batch["input_ids"].shape))

# Fail-fast generation smoke test before spending time on epoch 1.
smoke_example = dev_generation_dataset[0]
smoke_inputs = tokenizer(
    smoke_example["input_text"],
    return_tensors="pt",
    add_special_tokens=True,
)
smoke_inputs = {
    key: value.to(model.device)
    for key, value in smoke_inputs.items()
}
old_use_cache = model.config.use_cache
model.eval()
model.config.use_cache = True
try:
    with torch.inference_mode():
        model.generate(
            **smoke_inputs,
            max_new_tokens=4,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
finally:
    model.config.use_cache = old_use_cache
    model.train()

print("PASS: baseline generation smoke test")

The run records DEV predictions and the best baseline adapter.

In [ ]:
print(
    "Baseline early stopping:",
    f"patience={EARLY_STOPPING_PATIENCE},",
    f"threshold={EARLY_STOPPING_THRESHOLD},",
    f"grace_epochs={EARLY_STOPPING_GRACE_EPOCHS}",
)

## 13. Start training

In [ ]:
print("RESUME_FROM", checkpoint if checkpoint else "pretrained Qwen2.5-7B", flush=True)
try:
    if resume and resume.get("dev_pending"):
        trainer._load_from_checkpoint(str(checkpoint))
        trainer.state = TrainerState.load_from_json(str(checkpoint / "trainer_state.json"))
        trainer.evaluate()
        trainer.state.save_to_json(str(checkpoint / "trainer_state.json"))
        publish_checkpoint(trainer.state)
    finished = (baseline_selection_callback.stale_epochs["baseline"] >= EARLY_STOPPING_PATIENCE
                or (resume and resume["epoch"] >= MAX_EPOCHS))
    if not finished and time.monotonic() < deadline:
        trainer.train(resume_from_checkpoint=str(checkpoint) if checkpoint else None)
except DevBudgetReached:
    print("DEV reached the session budget. Partial predictions and saved checkpoints are available.", flush=True)


## 14. Report selected adapters and outputs

The callback has already saved the selected baseline adapter. Save the final training state separately for recovery.

In [ ]:
pointer = RUN_ROOT / "resume-state.json"
if pointer.exists():
    print(pointer.read_text(), flush=True)
